# 뉴스 카테고리 분류 (Naive Bayes)

이 노트북은 `news.csv` 데이터를 사용해 `TF-IDF + MultinomialNB` 모델을 학습하고 저장합니다.

아래 셀을 위에서부터 순서대로 실행하면 됩니다.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [2]:
BASE_DIR = Path.cwd()
DATA_PATH = "news.csv"
MODEL_PATH = "news_category_pipeline_nb.joblib"

In [3]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["headline", "category"]).copy()
df["headline"] = df["headline"].astype(str).str.strip()
df["category"] = df["category"].astype(str).str.strip()

df = df[(df["headline"] != "") & (df["category"] != "")]

print("데이터 개수:", len(df))
print("카테고리 개수:", df["category"].nunique())
df.head()

데이터 개수: 10000
카테고리 개수: 29


,headline,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df["headline"],
    df["category"],
    test_size=0.2,
    random_state=42,
    stratify=df["category"],
)

# Initialize and fit the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

Naive Bayes 테스트 정확도: 0.4960


In [14]:
# Initialize and fit the MultinomialNB model
model = LinearSVC()
model.fit(X_train_tfidf, y_train)

# Make predictions
preds = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, preds)

print(f"Naive Bayes 테스트 정확도: {acc:.4f}")

Naive Bayes 테스트 정확도: 0.6520


In [15]:
joblib.dump(model, MODEL_PATH)
print("모델 저장 완료:", MODEL_PATH)

모델 저장 완료: news_category_pipeline_nb.joblib


In [16]:
sample = "World leaders meet to discuss global economy"
sample_transformed = tfidf_vectorizer.transform([sample])
sample_pred = model.predict(sample_transformed)
print("샘플 헤드라인:", sample)
print("예측 카테고리:", sample_pred[0])

샘플 헤드라인: World leaders meet to discuss global economy
예측 카테고리: WORLD NEWS
